# (28) GIF: pvae dict elems (save MP4)

**Motivation**: Make $\Phi$ plots, turn to GIF. This notebook loads PNG figs and saves MP4 movie. <br>

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-vae/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-vae/figs')
tmp_dir = os.path.join(git_dir, 'jb-vae/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, 'PoissonVAE'))
from analysis.eval import sparse_score
from figures.fighelper import *
from main.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
from base.utils_model import load_model
from figures.imgs import plot_weights, make_grid

from moviepy.editor import ImageClip, concatenate_videoclips, ImageSequenceClip

import moviepy
moviepy.__version__

'1.0.3'

In [3]:
anim_dir = pjoin(fig_base_dir, 'animation')
os.makedirs(anim_dir, exist_ok=True)
print(os.listdir(anim_dir))

[
    'material',
    'pvae-phi-gif_dpi-300.mp4',
    'mnist-to-imgnet.mp4',
    'output_video.mp4',
    'mnist-to-imgnet_dpi-140.mp4',
    'mnist-to-imgnet_dpi-300.mp4',
    'mnist-to-imgnet_dpi-150.mp4'
]

## Step 2: MP4

Load PNG from animation dir, make movie

In [4]:
dpi = 600
project_name = 'pvae-phi-gif'

In [5]:
load_dir = pjoin(anim_dir, 'material', f"{project_name}_dpi-{dpi}")
filename_pattern = rf"{project_name}_t=(\d+)\.png"

image_files = []
for f in sorted(os.listdir(load_dir), key=alphanum_sort_key):
    match = re.match(filename_pattern, f)
    if match:
        t = int(match.group(1))
        image_files.append((t, os.path.join(load_dir, f)))
image_files.sort(key=lambda x: x[0])

In [6]:
image_files = [(t, f) for t, f in image_files if t % 10 == 0]

In [7]:
sorted_image_paths = [file_path for _, file_path in image_files]
n_frames = len(sorted_image_paths)
n_frames

300

In [8]:
base_interval = 0.027
durations = [base_interval] * n_frames

# for i in range(n_frames):
#     interval = base_interval / np.log(i + 2)
#     durations.append(interval)

In [9]:
clips = []

for t, image_path in tqdm(enumerate(sorted_image_paths), total=n_frames):
    clip = ImageClip(image_path).set_duration(durations[t])
    clips.append(clip)

100%|█████████████████████████████████████████| 300/300 [04:07<00:00,  1.21it/s]


In [10]:
pause_start_duration = 0.5   # pause on the first frame (seconds)
pause_end_duration = 2.0     # pause last frame (seconds)

start_pause_clip = ImageClip(sorted_image_paths[0]).set_duration(pause_start_duration)
end_pause_clip = ImageClip(sorted_image_paths[-1]).set_duration(pause_end_duration)

video = concatenate_videoclips([start_pause_clip] + clips + [end_pause_clip], method="compose")

In [11]:
%%time


vid_file = f'{project_name}_dpi-{dpi}.mp4'
vid_file = pjoin(anim_dir, vid_file)

video.write_videofile(
    filename=vid_file,
    fps=60,                 # Frames per second (adjust as needed)
    codec='libx264',        # Video codec
    bitrate='50000k',        # Bitrate for higher quality
    audio=False,            # No audio
    threads=16,             # Number of threads for encoding
    preset='medium',        # Encoding speed/quality trade-off
)

Moviepy - Building video /home/hadi/Dropbox/git/jb-vae/figs/animation/pvae-phi-gif_dpi-600.mp4.
Moviepy - Writing video /home/hadi/Dropbox/git/jb-vae/figs/animation/pvae-phi-gif_dpi-600.mp4



Moviepy - Done !
Moviepy - video ready /home/hadi/Dropbox/git/jb-vae/figs/animation/pvae-phi-gif_dpi-600.mp4
CPU times: user 16min 46s, sys: 9min 49s, total: 26min 35s
Wall time: 29min 11s


In [ ]:
"""frame_paths = [f for _, f in image_files]
clip = ImageSequenceClip(frame_paths, fps=120)

gif_file = f'{project_name}_dpi-{dpi}.gif'
gif_file = pjoin(anim_dir, gif_file)

clip.write_gif(gif_file, fps=120)"""